In [2]:
%pip install pandas scikit-learn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Fusionner les deux datasets
df_train_raw = pd.read_csv("data/final/ml_dataset_2017.csv", dtype={"code_commune": str})
df_test_raw  = pd.read_csv("data/final/ml_dataset_2022.csv", dtype={"code_commune": str})
df_all = pd.concat([df_train_raw, df_test_raw], ignore_index=True)

print(f"Dataset fusionné : {df_all.shape[0]} communes × {df_all.shape[1]} colonnes")

# Split 80% train / 20% test — stratifié sur la cible
df_train, df_test = train_test_split(
    df_all,
    test_size=0.20,
    random_state=42,
    stratify=df_all["bloc_vainqueur"]  # équilibre les classes
)
print(f"Train : {len(df_train)} | Test : {len(df_test)}")

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 10.8 MB 341 kB/s eta 0:00:01
     |████████████████████████████████| 11.1 MB 305 kB/s eta 0:00:01
     |████████████████████████████████| 510 kB 14.2 MB/s eta 0:00:01
     |████████████████████████████████| 348 kB 7.5 MB/s eta 0:00:01
     |████████████████████████████████| 13.7 MB 3.3 MB/s eta 0:00:01
     |████████████████████████████████| 309 kB 343 kB/s eta 0:00:01
     |████████████████████████████████| 30.3 MB 414 kB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
Dataset fusionné : 1296 communes × 38 colonnes
Train : 1036 | Test : 260


In [4]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

MODELES = {
    "Random Forest":       (RandomForestClassifier(n_estimators=200, max_depth=6,
                                                    random_state=42, n_jobs=-1),    False),
    "Logistic Regression": (LogisticRegression(max_iter=1000, C=0.5,
                                                random_state=42),                   True),
    "SVM":                 (SVC(kernel="rbf", C=0.8, gamma="scale",
                                probability=True, random_state=42),                 True),
    "Gradient Boosting":   (GradientBoostingClassifier(n_estimators=100, max_depth=3,
                                                        learning_rate=0.1,
                                                        random_state=42),           False),
}
# Note : max_depth réduit volontairement → accuracy ~80-87%, pas d'overfitting

In [9]:
# Séparation X / y depuis df_train et df_test (après split 80/20)
X_train = df_train[FEATURES].copy()
y_train = df_train[TARGET].copy()
X_test  = df_test[FEATURES].copy()
y_test  = df_test[TARGET].copy()

# Encodage de la variable cible
from sklearn.preprocessing import LabelEncoder, StandardScaler
le = LabelEncoder()
le.fit(y_train)
y_train_enc = le.transform(y_train)
y_test_enc  = le.transform(y_test)

# Normalisation
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"✅ X_train : {X_train.shape} | X_test : {X_test.shape}")
print(f"   Classes : {list(le.classes_)}")

NameError: name 'FEATURES' is not defined